In [29]:
pip install -q transformers sentencepiece torch scikit-learn emoji accelerate pandas numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 12.7 MB/s eta 0:00:00


In [30]:
# ======================================================
# 1. Install Libraries
# ======================================================

import re
import emoji
import pandas as pd
import numpy as np
import torch
import torch.nn as nn



from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

from transformers import XLMRobertaTokenizer, XLMRobertaModel
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report



from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [31]:
# ======================================================
# 2. Load Dataset
# ======================================================

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
BASE_DIR = "/content/drive/MyDrive/DravidianLangTech"

TRAIN_PATH = f"{BASE_DIR}/PS_train.csv"
DEV_PATH   = f"{BASE_DIR}/PS_dev.csv"
TEST_PATH  = f"{BASE_DIR}/PS_test_without_labels.csv"

In [5]:
import os
os.path.exists(TRAIN_PATH), os.path.exists(DEV_PATH), os.path.exists(TEST_PATH)


(True, True, True)

In [6]:
train_df = pd.read_csv(TRAIN_PATH)
dev_df   = pd.read_csv(DEV_PATH)
test_df  = pd.read_csv(TEST_PATH)

TEXT_COL  = "text"  if "text"  in train_df.columns else train_df.columns[0]
LABEL_COL = "label" if "label" in train_df.columns else train_df.columns[1]

print("TEXT_COL :", TEXT_COL)
print("LABEL_COL:", LABEL_COL)

TEXT_COL : content
LABEL_COL: labels


In [10]:
HASHTAG_KB = {
    "#DMK": "Dravida Munnetra Kazhagam ruling party Tamil Nadu",
    "#ADMK": "All India Anna Dravida Munnetra Kazhagam opposition party Tamil Nadu",
    "#AIADMK": "All India Anna Dravida Munnetra Kazhagam opposition party Tamil Nadu",
    "#BJP": "Bharatiya Janata Party Indian national political party",
    "#Congress": "Indian National Congress political party",
    "#NTK": "Naam Tamilar Katchi Tamil nationalist political party",

    "#Modi": "Prime Minister Narendra Modi leader of Bharatiya Janata Party",
    "#Stalin": "MK Stalin Chief Minister of Tamil Nadu Dravida Munnetra Kazhagam leader",
    "#EPS": "Edappadi Palaniswami AIADMK leader",
    "#OPS": "O Panneerselvam AIADMK leader",

    "#TamilNadu": "South Indian state Tamil Nadu politics",
    "#DravidianModel": "Dravidian political ideology Tamil Nadu governance",
    "#TNPolitics": "Tamil Nadu political discussion",

    "#Election": "political election campaign voting democracy",
    "#Vote": "voting political participation democracy",
    "#Politics": "political governance debate public opinion"
}

def expand_hashtags(text):
    tags = re.findall(r"#\w+", str(text))
    return str(text) + " " + " ".join(HASHTAG_KB[t] for t in tags if t in HASHTAG_KB)

for df in [train_df, dev_df, test_df]:
    df[TEXT_COL] = df[TEXT_COL].apply(expand_hashtags)

In [12]:
SARC_EMOJIS = {
    "😂","🤣","😆","😹",
    "😏","🙃","😒","😑",
    "😬","😜","😝","🤡",
    "🤦","🤦‍♂️","🤦‍♀️",
    "🤷","🤷‍♂️","🤷‍♀️",
    "🙄","😅","😓","😔",
    "😠","😡","🤬",
    "😤","😩","😫",
    "👀","🔥","💀"
}
def emoji_features(text):
    ems = [c for c in text if c in emoji.EMOJI_DATA]
    return [len(ems), sum(e in SARC_EMOJIS for e in ems)]

In [17]:
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2),
    min_df=2
)

X_tfidf_train = tfidf.fit_transform(train_df[TEXT_COL])
X_tfidf_dev   = tfidf.transform(dev_df[TEXT_COL])
X_tfidf_test  = tfidf.transform(test_df[TEXT_COL])

svd = TruncatedSVD(n_components=256, random_state=42)

X_train = svd.fit_transform(X_tfidf_train)
X_dev   = svd.transform(X_tfidf_dev)
X_test  = svd.transform(X_tfidf_test)

In [18]:
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2),
    min_df=2
)

X_tfidf_train = tfidf.fit_transform(train_df[TEXT_COL])
X_tfidf_dev   = tfidf.transform(dev_df[TEXT_COL])
X_tfidf_test  = tfidf.transform(test_df[TEXT_COL])

svd = TruncatedSVD(n_components=256, random_state=42)

X_train = svd.fit_transform(X_tfidf_train)
X_dev   = svd.transform(X_tfidf_dev)
X_test  = svd.transform(X_tfidf_test)

In [19]:
le = LabelEncoder()
y_train = le.fit_transform(train_df[LABEL_COL])
y_dev   = le.transform(dev_df[LABEL_COL])

class_weights = torch.tensor(
    compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train),
        y=y_train
    ),
    dtype=torch.float
)


In [20]:
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-base")

class TamilDataset(Dataset):
    def __init__(self, texts, tfidf_feats, labels=None):
        self.texts = texts
        self.tfidf = tfidf_feats
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )
        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "tfidf": torch.tensor(self.tfidf[idx], dtype=torch.float),
            "emoji": torch.tensor(emoji_features(self.texts[idx]), dtype=torch.float)
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [21]:
def mean_pooling(out, mask):
    tok = out.last_hidden_state
    mask = mask.unsqueeze(-1).float()
    return (tok * mask).sum(1) / mask.sum(1)

class HybridXLMR(nn.Module):
    def __init__(self, n_labels):
        super().__init__()
        self.encoder = XLMRobertaModel.from_pretrained("xlm-roberta-base")
        self.classifier = nn.Sequential(
            nn.Linear(768*2 + 256 + 2, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, n_labels)
        )

    def forward(self, input_ids, attention_mask, tfidf, emoji):
        out = self.encoder(input_ids, attention_mask)
        cls  = out.last_hidden_state[:,0]
        mean = mean_pooling(out, attention_mask)
        return self.classifier(torch.cat([cls, mean, tfidf, emoji], dim=1))


In [32]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HybridXLMR(len(le.classes_)).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device),
    label_smoothing=0.1
)

def set_encoder_trainable(model, n):
    for name, p in model.encoder.named_parameters():
        p.requires_grad = False
        if "layer." in name:
            if int(name.split("layer.")[1].split(".")[0]) >= 12 - n:
                p.requires_grad = True

train_ds = TamilDataset(train_df[TEXT_COL].tolist(), X_train, y_train)
dev_ds   = TamilDataset(dev_df[TEXT_COL].tolist(), X_dev, y_dev)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
dev_loader   = DataLoader(dev_ds, batch_size=32)

best_f1 = 0.0

for epoch in range(6):
    set_encoder_trainable(model, [1,2,4,6][min(epoch,3)])

    if epoch == 3:
        optimizer.param_groups[0]["lr"] = 8e-6

    model.train()
    for b in train_loader:
        optimizer.zero_grad()
        loss = criterion(
            model(
                b["input_ids"].to(device),
                b["attention_mask"].to(device),
                b["tfidf"].to(device),
                b["emoji"].to(device)
            ),
            b["labels"].to(device)
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    model.eval()
    preds, gold = [], []
    with torch.no_grad():
        for b in dev_loader:
            out = model(
                b["input_ids"].to(device),
                b["attention_mask"].to(device),
                b["tfidf"].to(device),
                b["emoji"].to(device)
            )
            preds += out.argmax(1).cpu().tolist()
            gold  += b["labels"].tolist()

    f1 = f1_score(gold, preds, average="macro")
    print(f"Epoch {epoch+1} | Macro-F1: {f1:.4f}")

    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "best_model.pt")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1 | Macro-F1: 0.1550
Epoch 2 | Macro-F1: 0.2857
Epoch 3 | Macro-F1: 0.3276
Epoch 4 | Macro-F1: 0.3477
Epoch 5 | Macro-F1: 0.3599
Epoch 6 | Macro-F1: 0.3972


In [33]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
preds, gold = [], []

with torch.no_grad():
    for b in dev_loader:
        out = model(
            b["input_ids"].to(device),
            b["attention_mask"].to(device),
            b["tfidf"].to(device),
            b["emoji"].to(device)
        )
        preds.extend(out.argmax(1).cpu().numpy())
        gold.extend(b["labels"].cpu().numpy())

# ---- Macro-F1 ----
macro_f1 = f1_score(gold, preds, average="macro")
print("Macro-F1 (DEV):", macro_f1)

# ---- Class-wise Metrics ----
print(classification_report(
    gold,
    preds,
    target_names=le.classes_,
    digits=2
))

Macro-F1 (DEV): 0.39715041046572047
                   precision    recall  f1-score   support

         Negative       0.21      0.29      0.25        51
          Neutral       0.32      0.17      0.22        84
None of the above       1.00      0.90      0.95        20
      Opinionated       0.45      0.44      0.45       153
         Positive       0.29      0.49      0.36        69
        Sarcastic       0.47      0.40      0.43       115
    Substantiated       0.14      0.12      0.12        52

         accuracy                           0.37       544
        macro avg       0.41      0.40      0.40       544
     weighted avg       0.38      0.37      0.37       544



In [34]:
test_ds = TamilDataset(test_df[TEXT_COL].tolist(), X_test)
test_loader = DataLoader(test_ds, batch_size=32)

model.load_state_dict(torch.load("best_model.pt", map_location=device))
model.eval()

preds = []
with torch.no_grad():
    for b in test_loader:
        preds += model(
            b["input_ids"].to(device),
            b["attention_mask"].to(device),
            b["tfidf"].to(device),
            b["emoji"].to(device)
        ).argmax(1).cpu().tolist()

test_df["prediction"] = le.inverse_transform(preds)
test_df.to_csv("submission.csv", index=False)

print("✅ submission.csv generated")

✅ submission.csv generated


In [35]:
torch.save(model.state_dict(), "best_model.pt")